In [ ]:
#SETUP AND IMPORTS
!pip install tensorflow numpy

import torch
import torch.nn as nn
import numpy as np
import urllib.request

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

Using: cuda


In [ ]:
#LOAD DATASET
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
    'shakespeare.txt'
)

text = open('shakespeare.txt', 'r').read()
print(f'Total characters: {len(text):,}')
print(text[:200])

Total characters: 1,115,394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [ ]:
#PREPROCESSING
chars    = sorted(set(text))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
# Use only first 200k characters instead of 1M
encoded  = np.array([char2idx[c] for c in text[:200_000]])

VOCAB_SIZE = len(chars)
SEQ_LEN    = 150
print(f'Vocab size: {VOCAB_SIZE}')

Vocab size: 65


In [ ]:
#DATA LOADER
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data    = torch.tensor(data, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, i):
        return self.data[i:i+self.seq_len], self.data[i+1:i+self.seq_len+1]

dataset = CharDataset(encoded, SEQ_LEN)
loader  = DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)
print(f'Total batches: {len(loader)}')

Total batches: 1561


In [ ]:
# BUILD LSTM MODEL
# Cell 5
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=512, num_layers=3):
        super().__init__()
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.fc    = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        out, hidden = self.lstm(self.embed(x), hidden)
        return self.fc(out), hidden

model = CharLSTM(VOCAB_SIZE).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Parameters: 5,558,977


In [ ]:
# TRAIN MODEL
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
criterion = nn.CrossEntropyLoss()
EPOCHS    = 15

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for i, (x, y) in enumerate(loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits, _ = model(x)
        loss = criterion(logits.view(-1, VOCAB_SIZE), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()
        total_loss += loss.item()

        if i % 500 == 0:
            print(f'  Epoch {epoch}/{EPOCHS} | Batch {i}/{len(loader)} | Loss: {loss.item():.4f}')

    print(f'Epoch {epoch} done — Avg Loss: {total_loss/len(loader):.4f}\n')

  Epoch 1/15 | Batch 0/1561 | Loss: 0.3526
  Epoch 1/15 | Batch 500/1561 | Loss: 0.3553
  Epoch 1/15 | Batch 1000/1561 | Loss: 0.3648
  Epoch 1/15 | Batch 1500/1561 | Loss: 0.3591
Epoch 1 done — Avg Loss: 0.3638

  Epoch 2/15 | Batch 0/1561 | Loss: 0.3663
  Epoch 2/15 | Batch 500/1561 | Loss: 0.3793
  Epoch 2/15 | Batch 1000/1561 | Loss: 0.3737
  Epoch 2/15 | Batch 1500/1561 | Loss: 0.3726
Epoch 2 done — Avg Loss: 0.3717

  Epoch 3/15 | Batch 0/1561 | Loss: 0.3767
  Epoch 3/15 | Batch 500/1561 | Loss: 0.3656
  Epoch 3/15 | Batch 1000/1561 | Loss: 0.3638
  Epoch 3/15 | Batch 1500/1561 | Loss: 0.3543
Epoch 3 done — Avg Loss: 0.3691

  Epoch 4/15 | Batch 0/1561 | Loss: 0.3581
  Epoch 4/15 | Batch 500/1561 | Loss: 0.3775
  Epoch 4/15 | Batch 1000/1561 | Loss: 0.3553
  Epoch 4/15 | Batch 1500/1561 | Loss: 0.3632
Epoch 4 done — Avg Loss: 0.3656

  Epoch 5/15 | Batch 0/1561 | Loss: 0.3662
  Epoch 5/15 | Batch 500/1561 | Loss: 0.3576
  Epoch 5/15 | Batch 1000/1561 | Loss: 0.3564
  Epoch 5/15 |

In [ ]:
def predict_next_char(seed, top_n=5, temperature=0.8):
    model.eval()
    x = torch.tensor([[char2idx[c] for c in seed]], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        logits, _ = model(x)
        logits     = logits[0, -1] / temperature
        probs      = torch.softmax(logits, dim=-1).cpu().numpy()

    top_idx   = probs.argsort()[::-1][:top_n]
    top_chars = [(idx2char[i], probs[i]) for i in top_idx]

    print(f"Seed: '{seed}'")
    print(f"Predicted next character: '{top_chars[0][0]}'")
    print()
    print(f"{'Rank':<6} {'Char':<10} {'Probability':<15} {'Bar'}")
    print("-" * 50)
    for rank, (char, prob) in enumerate(top_chars, 1):
        bar = '█' * int(prob * 50)
        print(f"{rank:<6} {repr(char):<10} {prob:<15.4f} {bar}")


# Try it
predict_next_char('ROMEO')
print()
predict_next_char('To be or not to')
print()
predict_next_char('JULIET:\nO ')


Seed: 'ROMEO'
Predicted next character: ' '

Rank   Char       Probability     Bar
--------------------------------------------------
1      ' '        0.8652          ███████████████████████████████████████████
2      'A'        0.0626          ███
3      'B'        0.0539          ██
4      'T'        0.0172          
5      'H'        0.0007          

Seed: 'To be or not to'
Predicted next character: ' '

Rank   Char       Probability     Bar
--------------------------------------------------
1      ' '        0.9998          █████████████████████████████████████████████████
2      'o'        0.0002          
3      'w'        0.0000          
4      'u'        0.0000          
5      'g'        0.0000          

Seed: 'JULIET:
O '
Predicted next character: 'p'

Rank   Char       Probability     Bar
--------------------------------------------------
1      'p'        0.6041          ██████████████████████████████
2      'h'        0.3683          ██████████████████
3      'c'      